# Todesfälle Stadt Zürich — Scraper

Interaktives Notebook zum Testen des Scrapers.
Liest die Daten über die AEM JSON-API von stadt-zuerich.ch.

In [ ]:
%pip install -q requests beautifulsoup4 pandas

In [ ]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE_URL = "https://www.stadt-zuerich.ch/de/lebenslagen/tod/todesfaelle"

MONTH_NAMES = [
    "januar", "februar", "maerz", "april", "mai", "juni",
    "juli", "august", "september", "oktober", "november", "dezember",
]
MONTH_NAME_TO_NUM = {name: i + 1 for i, name in enumerate(MONTH_NAMES)}


def fetch_json(url: str) -> dict:
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return resp.json()


def convert_date(date_str: str) -> str:
    """DD.MM.YYYY -> YYYY-MM-DD"""
    match = re.fullmatch(r"(\d{1,2})\.(\d{1,2})\.(\d{4})", date_str.strip())
    if match:
        day, month, year = match.groups()
        return f"{year}-{int(month):02d}-{int(day):02d}"
    return date_str

## 1. Verfügbare Jahre und Monate entdecken

In [ ]:
data = fetch_json(f"{BASE_URL}.1.json")
years = sorted([
    key for key, val in data.items()
    if isinstance(val, dict) and val.get("jcr:primaryType") == "cq:Page" and re.fullmatch(r"\d{4}", key)
])
print(f"Verfügbare Jahre: {years}")

In [ ]:
year_months = {}
for year in years:
    data = fetch_json(f"{BASE_URL}/{year}.1.json")
    months = sorted(
        [k for k, v in data.items() if isinstance(v, dict) and v.get("jcr:primaryType") == "cq:Page" and k in MONTH_NAME_TO_NUM],
        key=lambda m: MONTH_NAME_TO_NUM[m],
    )
    year_months[year] = months
    print(f"{year}: {months}")

## 2. Einzelnen Monat laden und inspizieren

Hier kannst du Jahr und Monat anpassen, um die Rohdaten zu prüfen.

In [ ]:
# --- Anpassen zum Testen ---
test_year = "2025"
test_month = "januar"
# ---------------------------

url = f"{BASE_URL}/{test_year}/{test_month}/_jcr_content/mainparsys.1.json"
parsys = fetch_json(url)

table_html = None
for key, val in parsys.items():
    if isinstance(val, dict) and "tableData" in val:
        table_html = val["tableData"]
        print(f"Tabelle gefunden in Komponente: {key}")
        print(f"Titel: {val.get('jcr:title', '–')}")
        break

if not table_html:
    print("WARNUNG: Keine Tabellendaten gefunden!")

In [ ]:
# Rohes HTML der Tabelle ansehen (erste 2000 Zeichen)
if table_html:
    print(table_html[:2000])

## 3. Tabelle parsen

In [ ]:
def parse_table(html: str, year: str, month: str) -> list[dict]:
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    if not table:
        return []

    rows = table.find_all("tr")
    header_cells = rows[0].find_all(["th", "td"])
    print(f"Header-Spalten: {[c.get_text(strip=True) for c in header_cells]}")
    print(f"Anzahl Zeilen (ohne Header): {len(rows) - 1}")

    month_num = MONTH_NAME_TO_NUM[month]
    todesmonat = f"{year}-{month_num:02d}"

    records = []
    for row in rows[1:]:
        cells = row.find_all("td")
        if len(cells) < 8:
            continue
        values = [c.get_text(strip=True) for c in cells]
        name, vorname, jahrgang, strasse, nummer, plz, ort, sterbedatum_raw = values[:8]
        records.append({
            "Todesmonat": todesmonat,
            "Name": name,
            "Vorname": vorname,
            "Jahrgang": jahrgang,
            "Strasse": strasse,
            "Nummer": nummer,
            "PLZ": plz,
            "Ort": ort,
            "Sterbedatum": convert_date(sterbedatum_raw),
        })
    return records


if table_html:
    records = parse_table(table_html, test_year, test_month)
    print(f"\n{len(records)} Einträge geparst")

In [ ]:
# Ergebnis als DataFrame anzeigen
if table_html:
    df = pd.DataFrame(records)
    print(f"Shape: {df.shape}")
    display(df.head(10))

## 4. Alle Monate laden

In [ ]:
all_records = []

for year, months in year_months.items():
    for month in months:
        print(f"Lade {year}/{month} ...", end=" ")
        url = f"{BASE_URL}/{year}/{month}/_jcr_content/mainparsys.1.json"
        try:
            parsys = fetch_json(url)
        except Exception as e:
            print(f"FEHLER: {e}")
            continue

        html = None
        for key, val in parsys.items():
            if isinstance(val, dict) and "tableData" in val:
                html = val["tableData"]
                break

        if not html:
            print("keine Tabelle")
            continue

        records = parse_table(html, year, month)
        print(f"{len(records)} Einträge")
        all_records.extend(records)

print(f"\nTotal: {len(all_records)} Einträge")

In [ ]:
df_all = pd.DataFrame(all_records)
df_all = df_all.sort_values(["Todesmonat", "Name", "Vorname"]).reset_index(drop=True)
print(f"Shape: {df_all.shape}")
display(df_all.head(10))
display(df_all.tail(10))

## 5. Schnelle Datenprüfung

In [ ]:
print("Einträge pro Monat:")
display(df_all.groupby("Todesmonat").size().rename("Anzahl").to_frame())

print(f"\nDatumsbereich Sterbedatum: {df_all['Sterbedatum'].min()} bis {df_all['Sterbedatum'].max()}")
print(f"Jahrgänge: {df_all['Jahrgang'].min()} bis {df_all['Jahrgang'].max()}")
print(f"Eindeutige PLZ: {sorted(df_all['PLZ'].unique())}")

## 6. Als CSV speichern

In [ ]:
from pathlib import Path

output = Path("data/todesfaelle_stadt_zuerich.csv")
output.parent.mkdir(parents=True, exist_ok=True)
df_all.to_csv(output, index=False, encoding="utf-8")
print(f"Gespeichert: {output} ({len(df_all)} Zeilen)")